本notebook用于机器学习中lightgbm和xgboost的学习
lightgbm底层设计：
1.histogram算法
即连续变量会被压缩成有限的箱子数，一般不超过255个，然后会对这些箱子的分界处进行切分，寻找最有效的一刀（最大的gain），这点其实跟woe略微有类似的点在。最终形成 所有指标 x 所有候选切点 寻找最有效的一刀/能增加最多gain的切点
2.leafwise, 即lightbgm如果切到了第一刀，eg：经营年限为三年，且在这里有最大的gain，那第二刀不同于第一刀的点在于，它会对比经营年限大于三年和经营年限小于三年的子叶各自在自己范围内的最优切点，比较两者gain的大小，选择gain最大的那个切点，而另一边整个树枝都会暂时保留但不继续向下生长，除非在后续对比中它其中含有最大的gain的切点
3.Boosting&gain：我们在找切点时，是在寻找最大的gain，如上所述。而在第一个切点成立后，基于第一个切点的划分逻辑，所有可能可行的所有切分点的gradient&hessian都改变了（gradient告诉lightgbm改的方向，hessian告诉改的大小，不引入更高阶的导数，那样计算成本过高，即泰勒展开的一阶二阶），我们会重新计算所有可能的选项，然后跟上一步操作一样，选择最大的gain，这个过程叫boosting。

4.模型复杂程度的控制（这里看下图）： 代码中用num_leaves来决定一棵树的的最大叶片数量，即模型的规则复杂程度和其带来的客户群切分数量。max_depth决定最大深度，即判断链条的深浅。并且这两个东西是有差别的，如图，每次选择切点后，最新的切点备选是所有最新的叶节点。min_data_in_leaf:每个客群最少的客户数，对比允许区分多少客群数量的num_leaves，它们的限制维度不同。learning rate：修正的小需要更多的树，修正的大则容易过拟合，是一个取舍问题，则引入了n estimators 它会绝对boosting修正多少轮次。 early stop：train ar肯定会一直提高，但如果validation ar先提高到某个无法上升的阶段了，则停止。
feature sampling：每颗树看多少个指标，通过feature fraction控制；bagging：每颗树使用的客户数量 bagging fraction控制。missing也会作为信息被利用，辅助左右判断 eg：企业有没有提供现金流数据
shap:相比于feature importance，告诉你客户级别上，指标的影响进行如果取加权平均则是全模型变量的重要性排名（对于变量平均对于客户预测结果的影响大小）

In [2]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split, ParameterSampler
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score


In [3]:
random_state = 42

In [4]:
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns',200)

In [5]:
#option + 右键可以快速拷贝绝对路径
df = pd.read_csv('/Users/ganlu/Desktop/home-credit-risk-english/data/application_train.csv')

In [6]:
X = df.drop(columns = ['TARGET','SK_ID_CURR'])
y = df['TARGET']

In [7]:
column_cn_map = {

    # =========================
    # ID / TARGET
    # =========================
    'SK_ID_CURR': '当前贷款申请ID',
    'TARGET': '是否违约',

    # =========================
    # 基本信息
    # =========================
    'NAME_CONTRACT_TYPE': '贷款合同类型',
    'CODE_GENDER': '性别',
    'FLAG_OWN_CAR': '是否有车',
    'FLAG_OWN_REALTY': '是否有房产',
    'CNT_CHILDREN': '子女数量',

    # =========================
    # 金额信息
    # =========================
    'AMT_INCOME_TOTAL': '总收入',
    'AMT_CREDIT': '贷款金额',
    'AMT_ANNUITY': '贷款年金/分期金额',
    'AMT_GOODS_PRICE': '商品价格',

    # =========================
    # 客户属性
    # =========================
    'NAME_TYPE_SUITE': '申请陪同人员类型',
    'NAME_INCOME_TYPE': '收入类型',
    'NAME_EDUCATION_TYPE': '教育程度',
    'NAME_FAMILY_STATUS': '婚姻家庭状态',
    'NAME_HOUSING_TYPE': '住房类型',

    # =========================
    # 地区 / 年龄 / 工作
    # =========================
    'REGION_POPULATION_RELATIVE': '所在地区相对人口规模',
    'DAYS_BIRTH': '距出生日期天数',
    'DAYS_EMPLOYED': '距开始工作天数',
    'DAYS_REGISTRATION': '距当前登记日期天数',
    'DAYS_ID_PUBLISH': '距身份证件更新天数',
    'OWN_CAR_AGE': '汽车车龄',

    # =========================
    # 联系方式
    # =========================
    'FLAG_MOBIL': '是否提供手机',
    'FLAG_EMP_PHONE': '是否提供单位电话',
    'FLAG_WORK_PHONE': '是否提供工作电话',
    'FLAG_CONT_MOBILE': '手机是否可联系',
    'FLAG_PHONE': '是否提供电话',
    'FLAG_EMAIL': '是否提供邮箱',

    # =========================
    # 职业 / 家庭
    # =========================
    'OCCUPATION_TYPE': '职业类型',
    'CNT_FAM_MEMBERS': '家庭成员数量',

    # =========================
    # 地区评级
    # =========================
    'REGION_RATING_CLIENT': '客户所在地区评级',
    'REGION_RATING_CLIENT_W_CITY': '客户所在地区评级_考虑城市',

    # =========================
    # 申请时间
    # =========================
    'WEEKDAY_APPR_PROCESS_START': '申请开始星期',
    'HOUR_APPR_PROCESS_START': '申请开始小时',

    # =========================
    # 居住地 / 工作地一致性
    # =========================
    'REG_REGION_NOT_LIVE_REGION': '登记地区与居住地区是否不同',
    'REG_REGION_NOT_WORK_REGION': '登记地区与工作地区是否不同',
    'LIVE_REGION_NOT_WORK_REGION': '居住地区与工作地区是否不同',

    'REG_CITY_NOT_LIVE_CITY': '登记城市与居住城市是否不同',
    'REG_CITY_NOT_WORK_CITY': '登记城市与工作城市是否不同',
    'LIVE_CITY_NOT_WORK_CITY': '居住城市与工作城市是否不同',

    # =========================
    # 工作单位
    # =========================
    'ORGANIZATION_TYPE': '工作单位类型',

    # =========================
    # 外部信用评分
    # =========================
    'EXT_SOURCE_1': '外部信用评分1',
    'EXT_SOURCE_2': '外部信用评分2',
    'EXT_SOURCE_3': '外部信用评分3',

    # =========================
    # 房产信息 AVG
    # =========================
    'APARTMENTS_AVG': '公寓面积指标_均值',
    'BASEMENTAREA_AVG': '地下室面积指标_均值',
    'YEARS_BEGINEXPLUATATION_AVG': '房屋投入使用年限指标_均值',
    'YEARS_BUILD_AVG': '房屋建造年限指标_均值',
    'COMMONAREA_AVG': '公共区域面积指标_均值',
    'ELEVATORS_AVG': '电梯指标_均值',
    'ENTRANCES_AVG': '入口数量指标_均值',
    'FLOORSMAX_AVG': '最高楼层指标_均值',
    'FLOORSMIN_AVG': '最低楼层指标_均值',
    'LANDAREA_AVG': '土地面积指标_均值',
    'LIVINGAPARTMENTS_AVG': '住宅公寓面积指标_均值',
    'LIVINGAREA_AVG': '居住面积指标_均值',
    'NONLIVINGAPARTMENTS_AVG': '非住宅公寓面积指标_均值',
    'NONLIVINGAREA_AVG': '非居住面积指标_均值',

    # =========================
    # 房产信息 MODE
    # =========================
    'APARTMENTS_MODE': '公寓面积指标_众数',
    'BASEMENTAREA_MODE': '地下室面积指标_众数',
    'YEARS_BEGINEXPLUATATION_MODE': '房屋投入使用年限指标_众数',
    'YEARS_BUILD_MODE': '房屋建造年限指标_众数',
    'COMMONAREA_MODE': '公共区域面积指标_众数',
    'ELEVATORS_MODE': '电梯指标_众数',
    'ENTRANCES_MODE': '入口数量指标_众数',
    'FLOORSMAX_MODE': '最高楼层指标_众数',
    'FLOORSMIN_MODE': '最低楼层指标_众数',
    'LANDAREA_MODE': '土地面积指标_众数',
    'LIVINGAPARTMENTS_MODE': '住宅公寓面积指标_众数',
    'LIVINGAREA_MODE': '居住面积指标_众数',
    'NONLIVINGAPARTMENTS_MODE': '非住宅公寓面积指标_众数',
    'NONLIVINGAREA_MODE': '非居住面积指标_众数',

    # =========================
    # 房产信息 MEDI
    # =========================
    'APARTMENTS_MEDI': '公寓面积指标_中位数',
    'BASEMENTAREA_MEDI': '地下室面积指标_中位数',
    'YEARS_BEGINEXPLUATATION_MEDI': '房屋投入使用年限指标_中位数',
    'YEARS_BUILD_MEDI': '房屋建造年限指标_中位数',
    'COMMONAREA_MEDI': '公共区域面积指标_中位数',
    'ELEVATORS_MEDI': '电梯指标_中位数',
    'ENTRANCES_MEDI': '入口数量指标_中位数',
    'FLOORSMAX_MEDI': '最高楼层指标_中位数',
    'FLOORSMIN_MEDI': '最低楼层指标_中位数',
    'LANDAREA_MEDI': '土地面积指标_中位数',
    'LIVINGAPARTMENTS_MEDI': '住宅公寓面积指标_中位数',
    'LIVINGAREA_MEDI': '居住面积指标_中位数',
    'NONLIVINGAPARTMENTS_MEDI': '非住宅公寓面积指标_中位数',
    'NONLIVINGAREA_MEDI': '非居住面积指标_中位数',

    # =========================
    # 其他房屋信息
    # =========================
    'FONDKAPREMONT_MODE': '房屋大修基金类型',
    'HOUSETYPE_MODE': '房屋类型',
    'TOTALAREA_MODE': '房屋总面积指标',
    'WALLSMATERIAL_MODE': '墙体材料类型',
    'EMERGENCYSTATE_MODE': '房屋是否处于紧急状态',

    # =========================
    # 社交圈违约信息
    # =========================
    'OBS_30_CNT_SOCIAL_CIRCLE': '社交圈30天逾期观察人数',
    'DEF_30_CNT_SOCIAL_CIRCLE': '社交圈30天逾期人数',
    'OBS_60_CNT_SOCIAL_CIRCLE': '社交圈60天逾期观察人数',
    'DEF_60_CNT_SOCIAL_CIRCLE': '社交圈60天逾期人数',

    # =========================
    # 手机信息
    # =========================
    'DAYS_LAST_PHONE_CHANGE': '距最近一次更换手机号天数',

    # =========================
    # 文件材料
    # =========================
    'FLAG_DOCUMENT_2': '是否提供文件2',
    'FLAG_DOCUMENT_3': '是否提供文件3',
    'FLAG_DOCUMENT_4': '是否提供文件4',
    'FLAG_DOCUMENT_5': '是否提供文件5',
    'FLAG_DOCUMENT_6': '是否提供文件6',
    'FLAG_DOCUMENT_7': '是否提供文件7',
    'FLAG_DOCUMENT_8': '是否提供文件8',
    'FLAG_DOCUMENT_9': '是否提供文件9',
    'FLAG_DOCUMENT_10': '是否提供文件10',
    'FLAG_DOCUMENT_11': '是否提供文件11',
    'FLAG_DOCUMENT_12': '是否提供文件12',
    'FLAG_DOCUMENT_13': '是否提供文件13',
    'FLAG_DOCUMENT_14': '是否提供文件14',
    'FLAG_DOCUMENT_15': '是否提供文件15',
    'FLAG_DOCUMENT_16': '是否提供文件16',
    'FLAG_DOCUMENT_17': '是否提供文件17',
    'FLAG_DOCUMENT_18': '是否提供文件18',
    'FLAG_DOCUMENT_19': '是否提供文件19',
    'FLAG_DOCUMENT_20': '是否提供文件20',
    'FLAG_DOCUMENT_21': '是否提供文件21',

    # =========================
    # 征信查询次数
    # =========================
    'AMT_REQ_CREDIT_BUREAU_HOUR': '近1小时征信查询次数',
    'AMT_REQ_CREDIT_BUREAU_DAY': '近1天征信查询次数',
    'AMT_REQ_CREDIT_BUREAU_WEEK': '近1周征信查询次数',
    'AMT_REQ_CREDIT_BUREAU_MON': '近1个月征信查询次数',
    'AMT_REQ_CREDIT_BUREAU_QRT': '近1季度征信查询次数',
    'AMT_REQ_CREDIT_BUREAU_YEAR': '近1年征信查询次数'
}

In [8]:
df_cn = df.rename(columns = column_cn_map)

In [9]:
df_cn 

,当前贷款申请ID,是否违约,贷款合同类型,性别,是否有车,是否有房产,子女数量,总收入,贷款金额,贷款年金/分期金额,商品价格,申请陪同人员类型,收入类型,教育程度,婚姻家庭状态,住房类型,所在地区相对人口规模,距出生日期天数,距开始工作天数,距当前登记日期天数,距身份证件更新天数,汽车车龄,是否提供手机,是否提供单位电话,是否提供工作电话,手机是否可联系,是否提供电话,是否提供邮箱,职业类型,家庭成员数量,客户所在地区评级,客户所在地区评级_考虑城市,申请开始星期,申请开始小时,登记地区与居住地区是否不同,登记地区与工作地区是否不同,居住地区与工作地区是否不同,登记城市与居住城市是否不同,登记城市与工作城市是否不同,居住城市与工作城市是否不同,工作单位类型,外部信用评分1,外部信用评分2,外部信用评分3,公寓面积指标_均值,地下室面积指标_均值,房屋投入使用年限指标_均值,房屋建造年限指标_均值,公共区域面积指标_均值,电梯指标_均值,入口数量指标_均值,最高楼层指标_均值,最低楼层指标_均值,土地面积指标_均值,住宅公寓面积指标_均值,居住面积指标_均值,非住宅公寓面积指标_均值,非居住面积指标_均值,公寓面积指标_众数,地下室面积指标_众数,房屋投入使用年限指标_众数,房屋建造年限指标_众数,公共区域面积指标_众数,电梯指标_众数,入口数量指标_众数,最高楼层指标_众数,最低楼层指标_众数,土地面积指标_众数,住宅公寓面积指标_众数,居住面积指标_众数,非住宅公寓面积指标_众数,非居住面积指标_众数,公寓面积指标_中位数,地下室面积指标_中位数,房屋投入使用年限指标_中位数,房屋建造年限指标_中位数,公共区域面积指标_中位数,电梯指标_中位数,入口数量指标_中位数,最高楼层指标_中位数,最低楼层指标_中位数,土地面积指标_中位数,住宅公寓面积指标_中位数,居住面积指标_中位数,非住宅公寓面积指标_中位数,非居住面积指标_中位数,房屋大修基金类型,房屋类型,房屋总面积指标,墙体材料类型,房屋是否处于紧急状态,社交圈30天逾期观察人数,社交圈30天逾期人数,社交圈60天逾期观察人数,社交圈60天逾期人数,距最近一次更换手机号天数,是否提供文件2,是否提供文件3,是否提供文件4,是否提供文件5,是否提供文件6,是否提供文件7,是否提供文件8,是否提供文件9,是否提供文件10,是否提供文件11,是否提供文件12,是否提供文件13,是否提供文件14,是否提供文件15,是否提供文件16,是否提供文件17,是否提供文件18,是否提供文件19,是否提供文件20,是否提供文件21,近1小时征信查询次数,近1天征信查询次数,近1周征信查询次数,近1个月征信查询次数,近1季度征信查询次数,近1年征信查询次数
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,351000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.018801,-9461,-637,-3648.0,-2120,NaN,1,1,0,1,1,0,Laborers,1.0,2,2,WEDNESDAY,10,0,0,0,0,0,0,Business Entity Type 3,0.083037,0.262949,0.139376,0.0247,0.0369,0.9722,0.6192,0.0143,0.00,0.0690,0.0833,0.1250,0.0369,0.0202,0.0190,0.0000,0.0000,0.0252,0.0383,0.9722,0.6341,0.0144,0.0000,0.0690,0.0833,0.1250,0.0377,0.0220,0.0198,0.0,0.0000,0.0250,0.0369,0.9722,0.6243,0.0144,0.00,0.0690,0.0833,0.1250,0.0375,0.0205,0.0193,0.0000,0.0000,reg oper account,block of flats,0.0149,"Stone, brick",No,2.0,2.0,2.0,2.0,-1134.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,1129500.0,Family,State servant,Higher education,Married,House / apartment,0.003541,-16765,-1188,-1186.0,-291,NaN,1,1,0,1,1,0,Core staff,2.0,1,1,MONDAY,11,0,0,0,0,0,0,School,0.311267,0.622246,NaN,0.0959,0.0529,0.9851,0.7960,0.0605,0.08,0.0345,0.2917,0.3333,0.0130,0.0773,0.0549,0.0039,0.0098,0.0924,0.0538,0.9851,0.8040,0.0497,0.0806,0.0345,0.2917,0.3333,0.0128,0.0790,0.0554,0.0,0.0000,0.0968,0.0529,0.9851,0.7987,0.0608,0.08,0.0345,0.2917,0.3333,0.0132,0.0787,0.0558,0.0039,0.0100,reg oper account,block of flats,0.0714,Block,No,1.0,0.0,1.0,0.0,-828.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,135000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.010032,-19046,-225,-4260.0,-2531,26.0,1,1,1,1,1,0,Laborers,1.0,2,2,MONDAY,9,0,0,0,0,0,0,Government,NaN,0.555912,0.729567,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,-815.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,297000.0,Unaccompanied,Working,Secondary / secondary special,Civil marriage,House / apartment,0.008019,-19005,-3039,-9833.0,-2437,NaN,1,1,0,1,0,0,Laborers,2.0,2,2,WEDNESDAY,17,0,0,0,0,0,0,Business Entity Type 3,NaN,0.650442,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,0.0,2.0,0.0,-617.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,513000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.028663,-19932,-3038,-4311.0,-3458,NaN,1,1,0,1,0,0,Core staff,1.0,2,2,THURSDAY,11,0,0,0,0,1,1,Religion,NaN,0.322738,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN

In [23]:
cat_cols = X.select_dtypes(include=['object','string']).columns
for col in cat_cols:
    X_train[col] = X_train[col].astype('category')
    X_selection_[col] = X_selection[col].astype('category')

In [24]:
from sklearn.pipeline import Pipeline
#pipeline用于串行计算，columntransformer在教不同类别的变量分别要做什么

preprocessor = ColumnTransformer([
    
    ('num',
     SimpleImputer(strategy='median'),
     num_cols),
    
    ('cat',
     Pipeline([
         ('imputer', SimpleImputer(strategy='most_frequent')),
         ('onehot', OneHotEncoder(
             handle_unknown='ignore',
             sparse_output=True
         ))
     ]),
     cat_cols)
])

NameError: name 'num_cols' is not defined

In [25]:
#stratify的目的是尽可能保证分层抽样好坏的比例
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=random_state
)

X_selection, X_test, y_selection, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=random_state
)

In [26]:
model = LGBMClassifier(
    objective = 'binary',
    n_estimators = 2000,
    learning_rate = 0.03,
    num_leaves = 31,
    random_state = random_state,
    n_jobs = -1,
    verbosity = -1
)

In [27]:
from lightgbm import LGBMClassifier, early_stopping
model.fit(X_train, y_train, eval_set = [(X_selection, y_selection)], eval_metric ='auc', 
          callbacks=[early_stopping(100)])

/opt/miniconda3/envs/quant-finance/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[476]	valid_0's auc: 0.756615	valid_0's binary_logloss: 0.246247


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.03
,n_estimators,2000
,subsample_for_bin,200000
,objective,'binary'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [31]:
from sklearn.metrics import roc_auc_score, roc_curve
import numpy as np
def evaluate_model(model, X_train, y_train, X_selection, y_selection, X_test, y_test):
    pred_train = model.predict_proba(X_train)[:,1]
    pred_selection = model.predict_proba(X_selection)[:,1]
    pred_test = model.predict_proba(X_test)[:,1]
    train_auc = roc_auc_score(y_train, pred_train)
    selection_auc = roc_auc_score(y_selection, pred_selection)
    test_auc = roc_auc_score(y_test, pred_test)
    train_ar = 2* train_auc - 1
    selection_ar = 2 * selection_auc - 1
    test_ar = 2 * test_auc - 1
    def get_ks(y, pred):
        fpr, tpr, _ = roc_curve(y, pred)
        return np.max(tpr - fpr)
    train_ks = get_ks(y_train, pred_train)
    selection_ks = get_ks(y_selection, pred_selection)
    test_ks = get_ks(y_test, pred_test)

    # 输出
    print('Train AUC:', train_auc)
    print('Selection AUC:', selection_auc)
    print('Test AUC:', test_auc)

    print('\nTrain AR:', train_ar)
    print('Selection AR:', selection_ar)
    print('Test AR:', test_ar)

    print('\nTrain KS:', train_ks)
    print('Selection KS:', selection_ks)
    print('Test KS:', test_ks)

    return {
        'train_auc': train_auc,
        'selection_auc': selection_auc,
        'test_auc': test_auc,
        'train_ar': train_ar,
        'selection_ar': selection_ar,
        'test_ar': test_ar,
        'train_ks': train_ks,
        'selection_ks': selection_ks,
        'test_ks': test_ks
    }

In [32]:
result = evaluate_model(model, X_train, y_train, X_selection, y_selection, X_test, y_test)

Train AUC: 0.8335056132612878
Selection AUC: 0.7566151549832837
Test AUC: 0.7596510724559367

Train AR: 0.6670112265225756
Selection AR: 0.5132303099665674
Test AR: 0.5193021449118733

Train KS: 0.505655251239475
Selection KS: 0.3774581060005963
Test KS: 0.39324555699793545


In [36]:
df

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,APARTMENTS_AVG,BASEMENTAREA_AVG,YEARS_BEGINEXPLUATATION_AVG,YEARS_BUILD_AVG,COMMONAREA_AVG,ELEVATORS_AVG,ENTRANCES_AVG,FLOORSMAX_AVG,FLOORSMIN_AVG,LANDAREA_AVG,LIVINGAPARTMENTS_AVG,LIVINGAREA_AVG,NONLIVINGAPARTMENTS_AVG,NONLIVINGAREA_AVG,APARTMENTS_MODE,BASEMENTAREA_MODE,YEARS_BEGINEXPLUATATION_MODE,YEARS_BUILD_MODE,COMMONAREA_MODE,ELEVATORS_MODE,ENTRANCES_MODE,FLOORSMAX_MODE,FLOORSMIN_MODE,LANDAREA_MODE,LIVINGAPARTMENTS_MODE,LIVINGAREA_MODE,NONLIVINGAPARTMENTS_MODE,NONLIVINGAREA_MODE,APARTMENTS_MEDI,BASEMENTAREA_MEDI,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BUILD_MEDI,COMMONAREA_MEDI,ELEVATORS_MEDI,ENTRANCES_MEDI,FLOORSMAX_MEDI,FLOORSMIN_MEDI,LANDAREA_MEDI,LIVINGAPARTMENTS_MEDI,LIVINGAREA_MEDI,NONLIVINGAPARTMENTS_MEDI,NONLIVINGAREA_MEDI,FONDKAPREMONT_MODE,HOUSETYPE_MODE,TOTALAREA_MODE,WALLSMATERIAL_MODE,EMERGENCYSTATE_MODE,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,351000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.018801,-9461,-637,-3648.0,-2120,NaN,1,1,0,1,1,0,Laborers,1.0,2,2,WEDNESDAY,10,0,0,0,0,0,0,Business Entity Type 3,0.083037,0.262949,0.139376,0.0247,0.0369,0.9722,0.6192,0.0143,0.00,0.0690,0.0833,0.1250,0.0369,0.0202,0.0190,0.0000,0.0000,0.0252,0.0383,0.9722,0.6341,0.0144,0.0000,0.0690,0.0833,0.1250,0.0377,0.0220,0.0198,0.0,0.0000,0.0250,0.0369,0.9722,0.6243,0.0144,0.00,0.0690,0.0833,0.1250,0.0375,0.0205,0.0193,0.0000,0.0000,reg oper account,block of flats,0.0149,"Stone, brick",No,2.0,2.0,2.0,2.0,-1134.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,1129500.0,Family,State servant,Higher education,Married,House / apartment,0.003541,-16765,-1188,-1186.0,-291,NaN,1,1,0,1,1,0,Core staff,2.0,1,1,MONDAY,11,0,0,0,0,0,0,School,0.311267,0.622246,NaN,0.0959,0.0529,0.9851,0.7960,0.0605,0.08,0.0345,0.2917,0.3333,0.0130,0.0773,0.0549,0.0039,0.0098,0.0924,0.0538,0.9851,0.8040,0.0497,0.0806,0.0345,0.2917,0.3333,0.0128,0.0790,0.0554,0.0,0.0000,0.0968,0.0529,0.9851,0.7987,0.0608,0.08,0.0345,0.2917,0.3333,0.0132,0.0787,0.0558,0.0039,0.0100,reg oper account,block of flats,0.0714,Block,No,1.0,0.0,1.0,0.0,-828.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,135000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.010032,-19046,-225,-4260.0,-2531,26.0,1,1,1,1,1,0,Laborers,1.0,2,2,MONDAY,9,0,0,0,0,0,0,Government,NaN,0.555912,0.729567,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,

xgboost 和 lightgbm的区分
1.我们已经聊过了lightbgm，它会沿着最有收益的切点不断疯狂生长，因此如果不好好设置参数很容易出现过拟合的情况，但xgboost一直都会认真考虑每一刀，找到每一个局部最优，换句话说，树的生成是十分充分的，而不是像lightgbm那样可能单方面的停止生长，只要它的gain是大于我们的最低要求的，这一刀就是会砍下去

In [38]:
%pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 11.0 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [40]:
from xgboost import XGBClassifier
xgb_model = XGBClassifier(
    objective = 'binary:logistic',
    n_estimators = 2000,
    learning_rate = 0.03,
    max_depth = 6,
    min_child_weight =1,
    subsample = 1.0,
    colsample_bytree = 1.0,
    reg_alpha = 0.0,
    reg_lambda = 1.0,
    eval_metric = 'auc',
    early_stopping_rounds = 100,
    random_state = 42,
    n_jobs = -1,
    tree_method='hist',
    enable_categorical = True
)
xgb_model.fit(X_train, y_train, eval_set = [(X_selection, y_selection)],
              verbose = False)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,1.0
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",100
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes fro

In [41]:
xgb_result = evaluate_model(
    xgb_model,
    X_train,
    y_train,
    X_selection,
    y_selection,
    X_test,
    y_test
)


Train AUC: 0.8422082951647478
Selection AUC: 0.7567942362315375
Test AUC: 0.759730995184992

Train AR: 0.6844165903294956
Selection AR: 0.513588472463075
Test AR: 0.519461990369984

Train KS: 0.5190822915689877
Selection KS: 0.37573592808384326
Test KS: 0.39245218752001954


In [46]:
#随机森林
#这里先不用pipeline，先用循环计算来理解
from sklearn.preprocessing import OneHotEncoder
cat_cols = X_train.select_dtypes(include = ['category', 'object']).columns
num_cols = X_train.select_dtypes(exclude = ['category', 'object']).columns
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
#在训练集上训练onehot规则后，再将训练集进行统一转换,规则确定后其他的就都可以用了
encoder.fit(X_train[cat_cols])
X_train_cat = encoder.transform(X_train[cat_cols])
X_selection_cat = encoder.transform(X_selection[cat_cols])
X_test_cat = encoder.transform(X_test[cat_cols])


In [52]:
#不用pipeline的方式就仅供参考了，理解即可
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
import pandas as pd


# 1. 找分类变量
cat_cols = X_train.select_dtypes(
    include=['category', 'object']
).columns


# 2. 找数值变量
num_cols = X_train.select_dtypes(
    exclude=['category', 'object']
).columns


# 3. 创建 OneHotEncoder
encoder = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False
)


# 4. 用训练集学习类别
encoder.fit(X_train[cat_cols])


# 5. 转换三个数据集
X_train_cat = encoder.transform(
    X_train[cat_cols]
)

X_selection_cat = encoder.transform(
    X_selection[cat_cols]
)

X_test_cat = encoder.transform(
    X_test[cat_cols]
)


# 6. 获取 One-Hot 后的列名
cat_feature_names = encoder.get_feature_names_out(cat_cols)


# 7. 转成 DataFrame
X_train_cat = pd.DataFrame(
    X_train_cat,
    columns=cat_feature_names,
    index=X_train.index
)

X_selection_cat = pd.DataFrame(
    X_selection_cat,
    columns=cat_feature_names,
    index=X_selection.index
)

X_test_cat = pd.DataFrame(
    X_test_cat,
    columns=cat_feature_names,
    index=X_test.index
)


# 8. 数值变量原样取出来
X_train_num = X_train[num_cols]
X_selection_num = X_selection[num_cols]
X_test_num = X_test[num_cols]


# 9. 拼接
X_train_processed = pd.concat(
    [X_train_num, X_train_cat],
    axis=1
)

X_selection_processed = pd.concat(
    [X_selection_num, X_selection_cat],
    axis=1
)

X_test_processed = pd.concat(
    [X_test_num, X_test_cat],
    axis=1
)


# 10. 建立随机森林
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)


# 11. 训练
rf.fit(
    X_train_processed,
    y_train
)


# 12. 预测
train_pred = rf.predict_proba(
    X_train_processed
)[:, 1]

selection_pred = rf.predict_proba(
    X_selection_processed
)[:, 1]

test_pred = rf.predict_proba(
    X_test_processed
)[:, 1]

KeyboardInterrupt: 

In [ ]:
#不用pipeline的方式就仅供参考了，理解即可
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
import pandas as pd

rf_preprocessor = ColumnTransformer(
    transformers = [('cat',OneHotEncoder(handle_unknown = 'ignore'), cat_cols)], remainder = 'passthrough'
)

rf_model = RandomForestClassifier(n_estimators = 500, max_depth=None, max_features='sqrt',
                                  bootstrap = True, class_weight = None, random_state=42, n_jobs=-1)

#这里的pipeline是一个加工装置，加工方法是前面定义好的preprocessor
rf_pipeline = Pipeline([('preprocessor', rf_preprocessor), ('model', rf_model)])

rf_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains 

In [55]:
rf_result = evaluate_model(rf_pipeline, X_train, y_train, X_selection, y_selection, X_test, y_test)

Train AUC: 1.0
Selection AUC: 0.7253111214112917
Test AUC: 0.727126324559094

Train AR: 1.0
Selection AR: 0.45062224282258345
Test AR: 0.45425264911818797

Train KS: 1.0
Selection KS: 0.33160683435623195
Test KS: 0.3442129991359822


In [56]:
!pip install catboost

  Using cached graphviz-0.21-py3-none-any.whl.metadata (12 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.9/28.9 MB 26.2 MB/s  0:00:01m0:00:0100:01
Using cached graphviz-0.21-py3-none-any.whl (47 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [catboost]1/2 [catboost]


In [57]:
from catboost import CatBoostClassifier
cat_cols = X_train.select_dtypes(include=['object','category']).columns.tolist()

In [62]:
for col in cat_cols:
    X_train[col] = X_train[col].astype('object').fillna('Missing').astype(str)
    X_selection[col] = X_selection[col].astype('object').fillna('Missing').astype(str)
    X_test[col] = X_test[col].astype('object').fillna('Missing').astype(str)
    

In [63]:
#cat_cols可以自动处理分类变量不需要像前面的随机森林那样使用one hot encoder
#thread_counts = n_jobs 即尽量使用全部的cpu， verbose代表100轮打印一次结果
cat_model = CatBoostClassifier(
    iterations = 500,
    learning_rate = 0.05,
    depth=6,
    loss_function ='Logloss',
    eval_metric = 'AUC',
    random_seed = 42,
    thread_count = -1,
    verbose = 100
)

cat_model.fit(X_train, y_train, cat_features = cat_cols, eval_set = (X_selection, y_selection))

0:	test: 0.6670601	best: 0.6670601 (0)	total: 166ms	remaining: 1m 22s
100:	test: 0.7467809	best: 0.7467809 (100)	total: 8.9s	remaining: 35.2s
200:	test: 0.7528186	best: 0.7528186 (200)	total: 17.6s	remaining: 26.2s
300:	test: 0.7560438	best: 0.7560438 (300)	total: 26.3s	remaining: 17.4s
400:	test: 0.7575347	best: 0.7575347 (400)	total: 35.9s	remaining: 8.87s
499:	test: 0.7583103	best: 0.7583274 (498)	total: 45.1s	remaining: 0us

bestTest = 0.7583274031
bestIteration = 498

Shrink model to first 499 iterations.


CatBoostClassifier(depth=6, eval_metric='AUC', iterations=500, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=100)

In [64]:
cat_result = evaluate_model(cat_model, X_train, y_train, X_selection, y_selection, X_test, y_test)

Train AUC: 0.7774728549510929
Selection AUC: 0.7583274031160219
Test AUC: 0.7617347629047484

Train AR: 0.5549457099021857
Selection AR: 0.5166548062320437
Test AR: 0.5234695258094968

Train KS: 0.4141325064152712
Selection KS: 0.38253055378076145
Test KS: 0.39397758726158677


对于 0–1 分类的 MLP，正向传播就是每层节点将上一层的输出进行加权求和并加上 bias，再经过 sigmoid 等激活函数形成新的隐藏特征；这些隐藏特征继续作为下一层输入，重复“加权求和 → 激活”，最终输出层通过 sigmoid 得到属于 1 类的预测概率。
反向传播则从预测结果与真实结果产生的最终误差信号开始，根据连接权重把误差逐层向前分配；每经过 sigmoid 节点，还要乘上 a(1-a)，表示这个节点当前对输入变化的敏感程度。由此得到各节点的误差信号，再结合进入该节点的输入算出每条连接权重的梯度。全部梯度计算完成以后，才统一乘以 learning rate 修改权重和 bias。使用更新后的参数重新正向传播，如此反复，最终使 Loss 不断下降。
而在真实的运用情景下，一般会用relu代替sigmoid
这个神经元正向是正数
→ 误差允许继续往前传
这个神经元正向被ReLU压成0
→ 这一轮误差信号基本被截断
z > 0
→ derivative = 1

z < 0
→ derivative = 0

In [73]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

cat_cols = X_train.select_dtypes(include = ['object', 'category']).columns
num_cols = X_train.select_dtypes(exclude = ['object', 'category']).columns
nn_prepocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy = 'median')), ('scaler', StandardScaler())]), num_cols),
    ('cat',Pipeline([('imputer',SimpleImputer(strategy = 'most_frequent')),('onehot',OneHotEncoder(handle_unknown = 'ignore'))]),cat_cols)
])

/var/folders/2v/__1b87yj6l9d7xgv06crr1hr0000gn/T/ipykernel_11299/3931292022.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include = ['object', 'category']).columns


In [74]:
nn_model = MLPClassifier(hidden_layer_sizes = (64, 32),
activation = 'relu', max_iter = 200, random_state = 42, verbose = True)
nn_pipeline = Pipeline([('preprocessor', nn_prepocessor), ('model',nn_model)])
nn_pipeline.fit(X_train, y_train)
nn_result = evaluate_model(nn_pipeline, X_train, y_train, X_selection, y_selection, X_test, y_test)

Iteration 1, loss = 0.26065396
Iteration 2, loss = 0.25075713
Iteration 3, loss = 0.24898849
Iteration 4, loss = 0.24754316
Iteration 5, loss = 0.24594171
Iteration 6, loss = 0.24430531
Iteration 7, loss = 0.24279885
Iteration 8, loss = 0.24092962
Iteration 9, loss = 0.23928753
Iteration 10, loss = 0.23806820
Iteration 11, loss = 0.23636021
Iteration 12, loss = 0.23455693
Iteration 13, loss = 0.23315001
Iteration 14, loss = 0.23175095
Iteration 15, loss = 0.23044173
Iteration 16, loss = 0.22930390
Iteration 17, loss = 0.22815294
Iteration 18, loss = 0.22692452
Iteration 19, loss = 0.22567434
Iteration 20, loss = 0.22475389
Iteration 21, loss = 0.22375130
Iteration 22, loss = 0.22269236
Iteration 23, loss = 0.22187493
Iteration 24, loss = 0.22085512
Iteration 25, loss = 0.22022740
Iteration 26, loss = 0.21905794
Iteration 27, loss = 0.21839143
Iteration 28, loss = 0.21769577
Iteration 29, loss = 0.21666966
Iteration 30, loss = 0.21625658
Iteration 31, loss = 0.21543312
Iteration 32, los

/opt/miniconda3/envs/quant-finance/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Train AUC: 0.896039326348853
Selection AUC: 0.6555957511974065
Test AUC: 0.6573914209148559

Train AR: 0.7920786526977059
Selection AR: 0.3111915023948131
Test AR: 0.31478284182971183

Train KS: 0.6203809764873837
Selection KS: 0.23281089159505336
Test KS: 0.23307346092210762


In [78]:
arks_table = pd.DataFrame(
    [result, xgb_result, rf_result, cat_result, nn_result],
    index=['LightGBM', 'XGBoost', 'Random Forest', 'CatBoost', 'MLP']
).reset_index(names='model')

arks_table

,model,train_auc,selection_auc,test_auc,train_ar,selection_ar,test_ar,train_ks,selection_ks,test_ks
0,LightGBM,0.833506,0.756615,0.759651,0.667011,0.513230,0.519302,0.505655,0.377458,0.393246
1,XGBoost,0.842208,0.756794,0.759731,0.684417,0.513588,0.519462,0.519082,0.375736,0.392452
2,Random Forest,1.000000,0.725311,0.727126,1.000000,0.450622,0.454253,1.000000,0.331607,0.344213
3,CatBoost,0.777473,0.758327,0.761735,0.554946,0.516655,0.523470,0.414133,0.382531,0.393978
4,MLP,0.896039,0.655596,0.657391,0.792079,0.311192,0.314783,0.620381,0.232811,0.233073


现在来看看逻辑回归如何，我们做一个非常完整的逻辑回归

In [10]:
import pandas as pd
import numpy as np

from scipy.stats import spearmanr

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [11]:
def prepare_lr_data(df):
    data = df.copy()
    # =========================
    # 1. 时间类变量转为更容易解释的“年”
    # =========================
    data['AGE_YEARS'] = -data['DAYS_BIRTH'] / 365.25
    # Home Credit 中 365243 是特殊值
    # 不把它当正常工作天数，而是留给 Missing 箱
    #-2111天代表贷款前2111天被雇佣，这里修正方向再转换为年，后续同理
    employed = data['DAYS_EMPLOYED'].replace(365243, np.nan)
    data['EMPLOYED_YEARS'] = -employed / 365.25
    data['REGISTRATION_YEARS'] = (
        -data['DAYS_REGISTRATION'] / 365.25
    )
    data['ID_PUBLISH_YEARS'] = (
        -data['DAYS_ID_PUBLISH'] / 365.25
    )
    data['PHONE_CHANGE_YEARS'] = (
        -data['DAYS_LAST_PHONE_CHANGE'] / 365.25
    )
    # =========================
    # 2. 一些有信用风险意义的比例变量
    # =========================
    #比率计算时，分母出现0/结果本身就有np.nan就返回np.nan
    def safe_div(a, b):
        result = a / b.replace(0, np.nan)
        return result.replace(
            [np.inf, -np.inf],
            np.nan
        )
    #贷款金额比年收入
    data['CREDIT_INCOME_RATIO'] = safe_div(
        data['AMT_CREDIT'],
        data['AMT_INCOME_TOTAL']
    )
    #还款金额比收入
    data['ANNUITY_INCOME_RATIO'] = safe_div(
        data['AMT_ANNUITY'],
        data['AMT_INCOME_TOTAL']
    )
    #贷款金额相对每期还款金额的大小
    data['CREDIT_ANNUITY_RATIO'] = safe_div(
        data['AMT_CREDIT'],
        data['AMT_ANNUITY']
    )
    #商品价格相对于融资金额的关系
    data['GOODS_CREDIT_RATIO'] = safe_div(
        data['AMT_GOODS_PRICE'],
        data['AMT_CREDIT']
    )
    #家庭总收入/家庭人数
    data['INCOME_PER_PERSON'] = safe_div(
        data['AMT_INCOME_TOTAL'],
        data['CNT_FAM_MEMBERS']
    )
    #工作经验/年龄
    data['EMPLOYED_AGE_RATIO'] = safe_div(
        data['EMPLOYED_YEARS'],
        data['AGE_YEARS']
    )
    # 已经转换成年，原始 DAYS 指标删掉，防止高度重复，后续相关性筛查就不用再做了
    data = data.drop(
        columns=[
            'DAYS_BIRTH',
            'DAYS_EMPLOYED',
            'DAYS_REGISTRATION',
            'DAYS_ID_PUBLISH',
            'DAYS_LAST_PHONE_CHANGE'
        ]
    )
    return data
lr_df = prepare_lr_data(df)
lr_df.shape

(307511, 128)

In [12]:
#对于要检查缺失的特征，是因为缺失可能本身代表着某种信息
check_missing = [
    'OWN_CAR_AGE',
    'OCCUPATION_TYPE',
    'EXT_SOURCE_1',
    'EXT_SOURCE_2',
    'EXT_SOURCE_3',
    'FONDKAPREMONT_MODE',
    'HOUSETYPE_MODE',
    'WALLSMATERIAL_MODE',
    'EMERGENCYSTATE_MODE',
    'AMT_REQ_CREDIT_BUREAU_YEAR'
]
missing_check = []
for col in check_missing:
    miss = df[col].isna()
    missing_check.append({
        'feature': col,
        'missing_rate': miss.mean(),
        'bad_rate_missing': df.loc[miss, 'TARGET'].mean(),
        'bad_rate_nonmissing': df.loc[~miss, 'TARGET'].mean()
    })
missing_check = pd.DataFrame(missing_check)
missing_check
#结论分析，三个和房地产有关的特征缺失高度重合，外部评级信息源三号表示没拿到评级信息的客户更危险
#amt_req_credit_burden_year即过去一年是否查询过征信报告，如果没有对应了显著上升的风险。
#occupation_type对应的结果如果职业类型为空反而更安全，缺失本身是一种低风险信号

,feature,missing_rate,bad_rate_missing,bad_rate_nonmissing
0,OWN_CAR_AGE,0.659908,0.085000,0.072441
1,OCCUPATION_TYPE,0.313455,0.065131,0.087851
2,EXT_SOURCE_1,0.563811,0.085195,0.074955
3,EXT_SOURCE_2,0.002146,0.078788,0.080733
4,EXT_SOURCE_3,0.198253,0.093119,0.077665
5,FONDKAPREMONT_MODE,0.683862,0.086188,0.068919
6,HOUSETYPE_MODE,0.501761,0.091512,0.069870
7,WALLSMATERIAL_MODE,0.508408,0.091281,0.069815
8,EMERGENCYSTATE_MODE,0.473983,0.092607,0.070025
9,AMT_REQ_CREDIT_BUREAU_YEAR,0.135016,0.103374,0.077194


In [13]:
#检查是否是数据问题，比如一个人有车但没有有车时间的记录
pd.crosstab(
    df['OWN_CAR_AGE'].isna(),
    df['FLAG_OWN_CAR']
)

FLAG_OWN_CAR,N,Y
OWN_CAR_AGE,,
False,0,104582
True,202924,5


In [14]:
random_state = 42
X_lr = lr_df.drop(
    columns=['TARGET', 'SK_ID_CURR'])
y_lr = lr_df['TARGET']
X_train_lr, X_temp_lr, y_train_lr, y_temp_lr = train_test_split(
    X_lr,
    y_lr,
    test_size=0.30,
    stratify=y_lr,
    random_state=random_state
)
X_selection_lr, X_test_lr, y_selection_lr, y_test_lr = train_test_split(
    X_temp_lr,
    y_temp_lr,
    test_size=0.50,
    stratify=y_temp_lr,
    random_state=random_state
)

In [15]:
MISSING_LABEL = '__MISSING__'
RARE_LABEL = '__RARE__'
def get_woe_table(
    labels,
    y,
    neutral_tiny_missing=True,
    tiny_missing_n=100
):

    temp = pd.DataFrame({
        'bin': labels.astype(object),
        'y': np.asarray(y)
    })
    result = (
        temp
        .groupby(
            'bin',
            dropna=False,
            observed=False
        )['y']
        .agg(['count', 'sum'])
        .rename(columns={'sum': 'bad'})
    )
    result['good'] = (
        result['count'] -
        result['bad']
    )
    total_bad = result['bad'].sum()
    total_good = result['good'].sum()

    k = len(result)
    # +0.5 防止 bad=0 / good=0
    result['bad_pct'] = (
        result['bad'] + 0.5
    ) / (
        total_bad + 0.5 * k
    )
    result['good_pct'] = (
        result['good'] + 0.5
    ) / (
        total_good + 0.5 * k
    )
    result['woe'] = np.log(
        result['bad_pct'] /
        result['good_pct']
    )
    # ============================
    # 极小 Missing 箱做审慎处理
    # ============================
    if (
        neutral_tiny_missing
        and MISSING_LABEL in result.index
        and result.loc[
            MISSING_LABEL,
            'count'
        ] < tiny_missing_n
    ):
        # 极少量缺失不让模型赋予极端风险
        result.loc[
            MISSING_LABEL,
            'woe'
        ] = 0

        result.loc[
            MISSING_LABEL,
            'bad_pct'
        ] = result.loc[
            MISSING_LABEL,
            'good_pct'
        ]
    result['iv_bin'] = (
        result['bad_pct'] -
        result['good_pct']
    ) * result['woe']
    result['bad_rate'] = (
        result['bad'] /
        result['count']
    )
    feature_iv = result['iv_bin'].sum()
    return (
        result.reset_index(),
        feature_iv
    )
def numeric_bin_labels(
    s,
    edges
):

    bins = pd.cut(
        s,
        bins=edges,
        include_lowest=True,
        duplicates='drop'
    )

    labels = bins.astype(object)

    labels[pd.isna(s)] = (
        MISSING_LABEL
    )

    return labels

In [16]:

#s为对应指标名，y为违约标签，最少保留两个箱子，否则会剔除指标，misisng最小100个，若过小则做保护性处理
def fit_numeric_woe(s, y, q=10, min_nonmissing_bins=2, tiny_missing_n=100):
    s = pd.to_numeric(s, errors='coerce')
    nonmissing = s.notna()
    #如果指标全为missing或者只有两个数值型变量
    if (nonmissing.sum() == 0 or s[nonmissing].nunique() < 2):
        return None
    # ============================
    # 1. 初始等频分箱
    # ============================
    #retbins为True时，会同时返回bin（category类型）和edges（array类型），并删除重复边界
    try:
        _, raw_edges = pd.qcut(
            s[nonmissing],
            q=min(q, s[nonmissing].nunique()),
            retbins=True,
            duplicates='drop')
    except:
    #nanquantile会求指定分位点并自动忽略nan，npunique会去掉重复值并排序
        raw_edges = np.unique(
            np.nanquantile(s[nonmissing],np.linspace(0, 1, min(q, 10) + 1))
        )
    raw_edges = np.unique(
        raw_edges.astype(float)
    )
    if len(raw_edges) < 3:
        return None
    edges = raw_edges.copy()
    # 防止 selection/test 出现训练集范围之外的值
    edges[0] = -np.inf
    edges[-1] = np.inf
    # ============================
    # 2. 计算普通箱坏样本率
    # ============================
    def get_ordered_stats(edges):
        bins = pd.cut(
            s[nonmissing],
            bins=edges,
            include_lowest=True,
            duplicates='drop'
        )
        temp = pd.DataFrame({
            'bin': bins,
            'y': np.asarray(y)[
                nonmissing.to_numpy()
            ]
        })
        stats = (
            temp
            .groupby(
                'bin',
                observed=True
            )['y']
            .agg(['count', 'mean'])
            .reset_index()
        )
        return stats
    initial_stats = get_ordered_stats(
        edges
    )
    if len(initial_stats) < 2:
        return None
  # ============================
    # 3. 自动判断指标方向
    # ============================
    #直接看箱违约率和箱序号自动判断是同向还是反向关系
    rho = spearmanr(
        np.arange(
            len(initial_stats)
        ),
        initial_stats['mean']
    ).statistic
    direction = (
        'increasing'
        if np.isnan(rho) or rho >= 0
        else 'decreasing'
    )
    # ============================
    # 4. 单调性合并
    # ============================
    while True:
        stats = get_ordered_stats(
            edges
        )
        bad_rates = (
            stats['mean']
            .to_numpy()
        )
        violation = None
        for i in range(
            1,
            len(bad_rates)
        ):
            if (
                direction == 'increasing'
                and
                bad_rates[i] <
                bad_rates[i-1]
            ):

                violation = i
                break
            if (
                direction == 'decreasing'
                and
                bad_rates[i] >
                bad_rates[i-1]
            ):

                violation = i
                break
        # 已经完全单调
        if violation is None:
            break
        # 如果已经只剩两个普通箱
        # 两箱天然可以构成一个方向
        if len(edges) <= 3:

            direction = (
                'increasing'
                if bad_rates[-1] >= bad_rates[0]
                else 'decreasing'
            )

            break
        # 合并：
        # 删除当前逆序箱与前一个箱之间的边界
        edges = np.delete(
            edges,
            violation
        )
    final_stats = get_ordered_stats(
        edges
    )
    # 与我们之前的规则一致：
    # 如果普通数据最后只剩一个箱，
    # 则这个指标基本失去连续区分能力
    if (
        len(final_stats) <
        min_nonmissing_bins
    ):

        return None
    # ============================
    # 5. Missing 单独加入
    # ============================
    labels = numeric_bin_labels(
        s,
        edges
    )
    table, iv = get_woe_table(
        labels,
        y,
        neutral_tiny_missing=True,
        tiny_missing_n=tiny_missing_n
    )
    woe_map = dict(
        zip(
            table['bin'],
            table['woe']
        )
    )
    return {
        'type': 'numeric',
        'edges': edges,
        'direction': direction,
        'table': table,
        'iv': float(iv),
        'woe_map': woe_map
    }

In [17]:
def fit_categorical_woe(
    s,
    y,
    rare_pct=0.005,
    rare_min_count=200,
    tiny_missing_n=100
):

    x = (
        s.astype(object)
        .where(
            s.notna(),
            MISSING_LABEL
        )
        .astype(str)
    )


    counts = x.value_counts(
        dropna=False
    )


    rare_cut = max(
        rare_min_count,
        int(
            len(x) *
            rare_pct
        )
    )


    rare_levels = (
        counts[
            (counts < rare_cut)
            &
            (counts.index != MISSING_LABEL)
        ]
        .index
        .tolist()
    )


    x_grouped = x.where(
        ~x.isin(
            rare_levels
        ),
        RARE_LABEL
    )


    table, iv = get_woe_table(
        x_grouped,
        y,
        neutral_tiny_missing=True,
        tiny_missing_n=tiny_missing_n
    )


    # 如果所有 rare 加起来仍然很少
    # 不让几十个客户获得极端 WOE
    if RARE_LABEL in table['bin'].values:

        idx = (
            table['bin']
            == RARE_LABEL
        )

        rare_count = (
            table.loc[
                idx,
                'count'
            ]
            .iloc[0]
        )


        if (
            rare_count <
            rare_min_count
        ):

            table.loc[
                idx,
                'woe'
            ] = 0

            table.loc[
                idx,
                'iv_bin'
            ] = 0

            iv = (
                table['iv_bin']
                .sum()
            )


    woe_map = dict(
        zip(
            table['bin'],
            table['woe']
        )
    )


    return {

        'type':
        'categorical',

        'rare_levels':
        set(rare_levels),

        'table':
        table,

        'iv':
        float(iv),

        'woe_map':
        woe_map
    }

In [18]:
#这个函数担当起一个分流器的作用，判断是数值类变量/类别变量
def fit_feature_woe(
    s,
    y,
    q=10
):

    nunique = s.nunique(
        dropna=True
    )


    if nunique < 2:
        return None


    is_categorical = (

        s.dtype == 'object'

        or

        str(
            s.dtype
        ).startswith(
            'category'
        )

        or

        nunique <= 10
    )


    if is_categorical:

        return fit_categorical_woe(
            s,
            y
        )


    return fit_numeric_woe(
        s,
        y,
        q=q
    )

In [20]:
def transform_feature_woe(
    s,
    spec
):

    if (
        spec['type']
        == 'numeric'
    ):

        labels = numeric_bin_labels(
            pd.to_numeric(
                s,
                errors='coerce'
            ),
            spec['edges']
        )


        return (
            pd.Series(
                labels,
                index=s.index
            )
            .map(
                spec['woe_map']
            )
            .fillna(0)
            .astype(float)
        )


    x = (
        s.astype(object)
        .where(
            s.notna(),
            MISSING_LABEL
        )
        .astype(str)
    )


    x = x.where(
        ~x.isin(
            spec['rare_levels']
        ),
        RARE_LABEL
    )


    return (
        x
        .map(
            spec['woe_map']
        )
        .fillna(0)
        .astype(float)
    )
def transform_woe_frame(
    X,
    specs,
    features
):

    result = pd.DataFrame(
        index=X.index
    )


    for feature in features:

        result[feature] = (
            transform_feature_woe(
                X[feature],
                specs[feature]
            )
        )


    return result

In [21]:
feature_candidates = []

basic_drop = []


for col in X_train_lr.columns:

    s = X_train_lr[col]


    nunique = s.nunique(
        dropna=True
    )

    missing_rate = (
        s.isna().mean()
    )

    top_frequency = (
        s.value_counts(
            dropna=False,
            normalize=True
        )
        .iloc[0]
    )


    if nunique < 2:

        basic_drop.append(
            [col, 'constant']
        )


    elif missing_rate > 0.95:

        basic_drop.append(
            [col, 'missing > 95%']
        )


    elif top_frequency > 0.995:

        basic_drop.append(
            [col, 'quasi constant > 99.5%']
        )


    else:

        feature_candidates.append(
            col
        )
basic_drop = pd.DataFrame(
    basic_drop,
    columns=[
        'feature',
        'reason'
    ]
)
#相关特镇大于99.5%都是同一个值
basic_drop

,feature,reason
0,FLAG_MOBIL,quasi constant > 99.5%
1,FLAG_CONT_MOBILE,quasi constant > 99.5%
2,FLAG_DOCUMENT_2,quasi constant > 99.5%
3,FLAG_DOCUMENT_4,quasi constant > 99.5%
4,FLAG_DOCUMENT_7,quasi constant > 99.5%
5,FLAG_DOCUMENT_9,quasi constant > 99.5%
6,FLAG_DOCUMENT_10,quasi constant > 99.5%
7,FLAG_DOCUMENT_11,quasi constant > 99.5%
8,FLAG_DOCUMENT_12,quasi constant > 99.5%
9,FLAG_DOCUMENT_13,quasi constant > 99.5%


In [22]:
woe_specs = {}

iv_result = []


for feature in feature_candidates:

    spec = fit_feature_woe(
        X_train_lr[feature],
        y_train_lr,
        q=10
    )


    if spec is None:

        continue


    woe_specs[feature] = spec


    iv_result.append({

        'feature':
        feature,

        'iv':
        spec['iv'],

        'type':
        spec['type'],

        'bins':
        len(
            spec['table']
        ),

        'direction':
        spec.get(
            'direction'
        ),

        'missing_rate':
        X_train_lr[
            feature
        ].isna().mean()
    })
iv_table = (
    pd.DataFrame(
        iv_result
    )
    .sort_values(
        'iv',
        ascending=False
    )
    .reset_index(
        drop=True
    )
)

iv_table

,feature,iv,type,bins,direction,missing_rate
0,EXT_SOURCE_3,0.326991,numeric,11,decreasing,0.198275
1,EXT_SOURCE_2,0.307641,numeric,11,decreasing,0.002156
2,EXT_SOURCE_1,0.153018,numeric,11,decreasing,0.563852
3,EMPLOYED_YEARS,0.112593,numeric,10,decreasing,0.180045
4,EMPLOYED_AGE_RATIO,0.091479,numeric,10,decreasing,0.180045
5,AGE_YEARS,0.086219,numeric,10,decreasing,0.000000
6,GOODS_CREDIT_RATIO,0.068906,numeric,8,decreasing,0.000869
7,NAME_INCOME_TYPE,0.054586,categorical,5,NaN,0.000000
8,AMT_GOODS_PRICE,0.052167,numeric,5,decreasing,0.000869
9,REGION_RATING_CLIENT_W_CITY,0.051989,categorical,3,NaN,0.000000


In [23]:
single_ar_map = {}


for feature in woe_specs:

    temp_woe = (
        transform_feature_woe(
            X_train_lr[feature],
            woe_specs[feature]
        )
    )


    auc = roc_auc_score(
        y_train_lr,
        temp_woe
    )


    single_ar_map[
        feature
    ] = 2 * auc - 1
iv_table['single_ar'] = (
    iv_table['feature']
    .map(
        single_ar_map
    )
)

In [24]:
iv_table[
    [
        'feature',
        'iv',
        'single_ar',
        'missing_rate',
        'bins',
        'direction'
    ]
].head(30)
iv_min = 0.02


iv_features = (
    iv_table
    .loc[
        iv_table['iv']
        >= iv_min,
        'feature'
    ]
    .tolist()
)


len(iv_features)

51

In [25]:
X_train_woe = transform_woe_frame(
    X_train_lr,
    woe_specs,
    iv_features
)


X_selection_woe = transform_woe_frame(
    X_selection_lr,
    woe_specs,
    iv_features
)


X_test_woe = transform_woe_frame(
    X_test_lr,
    woe_specs,
    iv_features
)

In [26]:
def correlation_prune(
    X_woe,
    feature_info,
    threshold=0.7,
    ar_gap=0.03
):

    info = (
        feature_info
        .set_index(
            'feature'
        )
    )


    corr = (
        X_woe
        .corr()
        .abs()
    )


    features = list(
        X_woe.columns
    )


    drop = set()

    decisions = []


    def choose(
        feature_a,
        feature_b
    ):

        ar_a = info.loc[
            feature_a,
            'single_ar'
        ]

        ar_b = info.loc[
            feature_b,
            'single_ar'
        ]


        miss_a = info.loc[
            feature_a,
            'missing_rate'
        ]

        miss_b = info.loc[
            feature_b,
            'missing_rate'
        ]


        iv_a = info.loc[
            feature_a,
            'iv'
        ]

        iv_b = info.loc[
            feature_b,
            'iv'
        ]


        # ========================
        # AR 差异明显
        # ========================

        if (
            abs(ar_a - ar_b)
            >= ar_gap
        ):

            keep = (
                feature_a
                if ar_a >= ar_b
                else feature_b
            )

            reason = (
                'higher single AR'
            )


        # ========================
        # AR 相近，看缺失率
        # ========================

        elif (
            abs(
                miss_a -
                miss_b
            )
            > 1e-12
        ):

            keep = (
                feature_a
                if miss_a <= miss_b
                else feature_b
            )

            reason = (
                'AR close, lower missing'
            )


        else:

            keep = (
                feature_a
                if iv_a >= iv_b
                else feature_b
            )

            reason = (
                'higher IV'
            )


        remove = (
            feature_b
            if keep == feature_a
            else feature_a
        )


        return (
            keep,
            remove,
            reason
        )


    for i, feature_a in enumerate(
        features
    ):

        if feature_a in drop:
            continue


        for feature_b in features[
            i + 1:
        ]:

            if feature_b in drop:
                continue


            if (
                corr.loc[
                    feature_a,
                    feature_b
                ]
                >
                threshold
            ):

                keep, remove, reason = choose(
                    feature_a,
                    feature_b
                )


                drop.add(
                    remove
                )


                decisions.append({

                    'feature_a':
                    feature_a,

                    'feature_b':
                    feature_b,

                    'corr':
                    corr.loc[
                        feature_a,
                        feature_b
                    ],

                    'keep':
                    keep,

                    'drop':
                    remove,

                    'reason':
                    reason
                })


                if (
                    remove
                    == feature_a
                ):

                    break


    final_features = [
        feature
        for feature in features
        if feature not in drop
    ]


    decisions = pd.DataFrame(
        decisions
    )


    return (
        final_features,
        decisions
    )

In [27]:
corr_features, corr_decisions = correlation_prune(
    X_train_woe,
    iv_table,
    threshold=0.7,
    ar_gap=0.03
)

In [28]:
def calculate_vif(
    X
):

    X_const = sm.add_constant(
        X,
        has_constant='add'
    )


    values = (
        X_const
        .to_numpy(
            dtype=float
        )
    )


    result = []


    for i, feature in enumerate(
        X.columns,
        start=1
    ):

        result.append({

            'feature':
            feature,

            'VIF':
            variance_inflation_factor(
                values,
                i
            )
        })


    return (
        pd.DataFrame(
            result
        )
        .sort_values(
            'VIF',
            ascending=False
        )
    )

In [31]:
def fit_statsmodels_logit(
    X,
    y,
    features
):

    X_model = sm.add_constant(
        X[features],
        has_constant='add'
    )


    model = sm.Logit(
        y.astype(float),
        X_model.astype(float)
    )


    result = model.fit(
        disp=0,
        maxiter=200
    )


    return result

In [32]:
def refine_logit_features(
    X,
    y,
    features,
    p_threshold=0.05,
    vif_threshold=10,
    enforce_positive=True
):

    selected = list(
        features
    )

    history = []


    while True:

        # ==========================
        # 1. VIF
        # ==========================

        vif = calculate_vif(
            X[selected]
        )


        if (
            len(vif) > 0
            and
            vif.iloc[0]['VIF']
            >
            vif_threshold
        ):

            remove = (
                vif.iloc[0][
                    'feature'
                ]
            )


            history.append({

                'feature':
                remove,

                'reason':
                'VIF',

                'value':
                vif.iloc[0]['VIF']
            })


            selected.remove(
                remove
            )

            continue


        # ==========================
        # 2. Logistic
        # ==========================

        result = fit_statsmodels_logit(
            X,
            y,
            selected
        )


        # ==========================
        # 3. P-value
        # ==========================

        pvalues = (
            result
            .pvalues
            .drop('const')
        )


        if (
            pvalues.max()
            >
            p_threshold
        ):

            remove = (
                pvalues
                .idxmax()
            )


            history.append({

                'feature':
                remove,

                'reason':
                'P-value',

                'value':
                pvalues.max()
            })


            selected.remove(
                remove
            )

            continue


        # ==========================
        # 4. 系数方向
        # ==========================

        if enforce_positive:

            coefficients = (
                result
                .params
                .drop('const')
            )


            negative = (
                coefficients[
                    coefficients < 0
                ]
            )


            if len(negative) > 0:

                remove = (
                    negative
                    .idxmin()
                )


                history.append({

                    'feature':
                    remove,

                    'reason':
                    'negative coefficient',

                    'value':
                    negative.min()
                })


                selected.remove(
                    remove
                )

                continue


        break


    history = pd.DataFrame(
        history
    )


    return (
        selected,
        result,
        history
    )

In [33]:
final_features, sm_result, selection_history = refine_logit_features(
    X_train_woe,
    y_train_lr,
    corr_features,
    p_threshold=0.05,
    vif_threshold=10,
    enforce_positive=True
)

In [ ]:
#逻辑回归实际入模指标
final_features

['EXT_SOURCE_3',
 'EXT_SOURCE_2',
 'EXT_SOURCE_1',
 'EMPLOYED_YEARS',
 'GOODS_CREDIT_RATIO',
 'NAME_INCOME_TYPE',
 'REGION_RATING_CLIENT_W_CITY',
 'NAME_EDUCATION_TYPE',
 'PHONE_CHANGE_YEARS',
 'CREDIT_ANNUITY_RATIO',
 'CODE_GENDER',
 'ID_PUBLISH_YEARS',
 'FLAG_DOCUMENT_3',
 'REGISTRATION_YEARS',
 'REG_CITY_NOT_LIVE_CITY',
 'NAME_FAMILY_STATUS',
 'EMERGENCYSTATE_MODE',
 'OWN_CAR_AGE']

In [35]:
final_parameter_table = pd.DataFrame({

    'feature':
    final_features,

    'coefficient':
    sm_result.params[
        final_features
    ].values,

    'p_value':
    sm_result.pvalues[
        final_features
    ].values

})

In [36]:
final_parameter_table

,feature,coefficient,p_value
0,EXT_SOURCE_3,0.834946,0.000000e+00
1,EXT_SOURCE_2,0.732449,0.000000e+00
2,EXT_SOURCE_1,0.467242,6.627466e-93
3,EMPLOYED_YEARS,0.365208,1.585238e-34
4,GOODS_CREDIT_RATIO,0.691492,3.255017e-111
5,NAME_INCOME_TYPE,0.145246,5.914231e-04
6,REGION_RATING_CLIENT_W_CITY,0.298194,2.385905e-14
7,NAME_EDUCATION_TYPE,0.600135,5.018977e-49
8,PHONE_CHANGE_YEARS,0.161276,1.154683e-04
9,CREDIT_ANNUITY_RATIO,0.477029,1.505268e-27


In [37]:
final_vif = calculate_vif(
    X_train_woe[
        final_features
    ]
)

In [38]:
final_vif

,feature,VIF
3,EMPLOYED_YEARS,1.408835
5,NAME_INCOME_TYPE,1.370982
1,EXT_SOURCE_2,1.202058
6,REGION_RATING_CLIENT_W_CITY,1.157251
10,CODE_GENDER,1.124997
2,EXT_SOURCE_1,1.119213
12,FLAG_DOCUMENT_3,1.117453
11,ID_PUBLISH_YEARS,1.097694
13,REGISTRATION_YEARS,1.089570
7,NAME_EDUCATION_TYPE,1.086248


In [39]:
final_parameter_table = (
    final_parameter_table
    .merge(
        iv_table[
            [
                'feature',
                'iv',
                'single_ar',
                'missing_rate'
            ]
        ],
        on='feature',
        how='left'
    )
    .merge(
        final_vif,
        on='feature',
        how='left'
    )
)

In [40]:
final_parameter_table

,feature,coefficient,p_value,iv,single_ar,missing_rate,VIF
0,EXT_SOURCE_3,0.834946,0.000000e+00,0.326991,0.313519,0.198275,1.060012
1,EXT_SOURCE_2,0.732449,0.000000e+00,0.307641,0.306048,0.002156,1.202058
2,EXT_SOURCE_1,0.467242,6.627466e-93,0.153018,0.177601,0.563852,1.119213
3,EMPLOYED_YEARS,0.365208,1.585238e-34,0.112593,0.187387,0.180045,1.408835
4,GOODS_CREDIT_RATIO,0.691492,3.255017e-111,0.068906,0.138043,0.000869,1.056546
5,NAME_INCOME_TYPE,0.145246,5.914231e-04,0.054586,0.118303,0.000000,1.370982
6,REGION_RATING_CLIENT_W_CITY,0.298194,2.385905e-14,0.051989,0.098494,0.000000,1.157251
7,NAME_EDUCATION_TYPE,0.600135,5.018977e-49,0.048592,0.091031,0.000000,1.086248
8,PHONE_CHANGE_YEARS,0.161276,1.154683e-04,0.044289,0.110557,0.000005,1.075010
9,CREDIT_ANNUITY_RATIO,0.477029,1.505268e-27,0.039216,0.096570,0.000037,1.065891


In [41]:
lr_model = LogisticRegression(
    penalty=None,
    solver='lbfgs',
    max_iter=1000
)

In [42]:
lr_model.fit(
    X_train_woe[
        final_features
    ],
    y_train_lr
)

/opt/miniconda3/envs/quant-finance/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",None
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass`

In [43]:
lr_result = evaluate_model(
    lr_model,

    X_train_woe[
        final_features
    ],
    y_train_lr,

    X_selection_woe[
        final_features
    ],
    y_selection_lr,

    X_test_woe[
        final_features
    ],
    y_test_lr
)

NameError: name 'evaluate_model' is not defined